In [8]:
# CELL 1: Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import json
import warnings
warnings.filterwarnings('ignore')
from transformers import ViTForImageClassification, ViTFeatureExtractor

# Configuration
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: mps


## Load Database

In [9]:
from dataset import CarDamageDataset

data = torch.load("../../../development/database/cars_damage_dataset/database/data_preparation_outputs.pth", weights_only=False)
train_loader = data['train_loader']
val_loader = data['val_loader']
label_encoder = data['label_encoder']
class_mapping = data['class_mapping']

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Training batches: 21
Validation batches: 6
Number of classes: 3


## Setup Vision Transformer

In [10]:
model_name = "google/vit-base-patch16-224-in21k"
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_),
    ignore_mismatched_sizes=True
)
model = model.to(DEVICE)

print(f"Model loaded: {model_name}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: google/vit-base-patch16-224-in21k
Number of classes: 3


## Setup Optimizer and Scheduler

In [11]:
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
criterion = nn.CrossEntropyLoss()

print("Setup complete. Ready to train the model.")

Setup complete. Ready to train the model.


## Training function

In [12]:
from tqdm.auto import tqdm

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, labels in progress_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        progress_bar.set_postfix(loss=loss.item())

    accuracy = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), accuracy

## Validation function

In [ ]:
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(loader, desc="Validating", leave=False)

    with torch.no_grad():
        for images, labels in progress_bar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            progress_bar.set_postfix(loss=loss.item())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    return total_loss / len(loader), accuracy, precision, recall, f1

: 

## Training Loop with Early Stopping

In [ ]:
EPOCHS = 20
PATIENCE = 5
best_val_acc = 0
patience_counter = 0
metrics_history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_precision, val_recall, val_f1 = validate_epoch(model, val_loader, criterion)

    scheduler.step()

    metrics_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'val_precision': val_precision,
        'val_recall': val_recall,
        'val_f1': val_f1
    })

    print(f"Epoch {epoch + 1}/{EPOCHS} - ")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"  Val Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "vit_damage_classifier.pth")
        print(" ✅ Best model saved.")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  ⏹️ Early stopping triggered at epoch {epoch+1}")
            break

# Save metrics history to a JSON file
pd.DataFrame(metrics_history).to_csv("training_metrics.csv", index=False)

Training:   0%|          | 0/21 [00:01<?, ?it/s]

Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps


Validating:   0%|          | 0/6 [00:02<?, ?it/s]

Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps
Using device: mps


In [ ]:
# Load best model
model.load_state_dict(torch.load("vit_damage_classifier.pth"))
model.eval()
print("Best model loaded for evaluation.")

In [ ]:
# Evaluate on validation set
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(pixel_values=images)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))

## Confusion matrix visualization

In [ ]:
# CELL 10: Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix - Damage Classification')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate and save final metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

final_metrics = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1
}

with open('final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print(f"Final Metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1 Score: {f1:.4f}")

In [ ]:
# CELL 11: Training History Visualization
metrics_df = pd.DataFrame(metrics_history)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(metrics_df['epoch'], metrics_df['train_loss'], label='Train Loss', marker='o')
axes[0].plot(metrics_df['epoch'], metrics_df['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(metrics_df['epoch'], metrics_df['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(metrics_df['epoch'], metrics_df['val_acc'], label='Val Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

##